# Mini Project 5 — Deploy AI System on Azure
## BSI IT Helpdesk AI Assistant

Pipeline: Ticket → Classifier → RAG Retrieval → GPT-4.1-mini → Cited Response

**Checklist submission:**
- [ ] Source code (notebook ini)
- [ ] Evaluation report (CSV dari AI Foundry)
- [ ] Cost & performance sheet (XLSX)
- [ ] Architecture diagram (PNG/PDF)

---
### Prerequisites
```bash
pip install azure-search-documents azure-ai-ml azure-identity openai tiktoken pandas openpyxl
```

## CHAPTER 1 — Foundation & Setup (Hour 1)

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────
# Ganti nilai di bawah sesuai resource Azure kamu

SUBSCRIPTION_ID   = "<your-subscription-id>"
RESOURCE_GROUP    = "rg-bsi-itdesk-<your-initials>"   # C1: sesuai instruksi
LOCATION          = "indonesiacentral"                 # default Indonesia Central

# Azure OpenAI
AOAI_ENDPOINT     = "https://<your-aoai>.openai.azure.com/"
AOAI_KEY          = "<your-aoai-key>"
CHAT_DEPLOY       = "gpt-4.1-mini"                    # RECOMMENDED (cost 8, quality 94)
EMBED_DEPLOY      = "text-embedding-3-small"

# Azure AI Search
SEARCH_ENDPOINT   = "https://<your-search>.search.windows.net"
SEARCH_KEY        = "<your-search-admin-key>"
INDEX_NAME        = "bsi-itdesk-knowledge"

# Azure ML
AML_WORKSPACE     = "<your-aml-workspace>"
AML_ENDPOINT_NAME = "bsi-ticket-classifier"

# Blob Storage
BLOB_CONN_STR     = "<your-blob-connection-string>"
BLOB_CONTAINER    = "itdesk-docs"

print("Config loaded ✓")

In [ ]:
# ── MOCK DATA: SOP Documents ────────────────────────────────────────────
# Simulasi dokumen SOP yang akan di-index ke Azure AI Search

SOP_DOCS = [
    {
        "id": "sop-001",
        "title": "Password Reset & MFA Token",
        "category": "access",
        "content": """
        SOP: Cara reset password dan MFA token BSI.
        1. Buka portal self-service di https://aka.ms/sspr
        2. Klik 'Forgot my password' dan masukkan email BSI kamu
        3. Pilih metode verifikasi (email / SMS)
        4. Untuk reset MFA token: login ke https://aka.ms/mfasetup
        5. Klik 'Security info' → 'Add method' → pilih Authenticator App
        6. Scan QR code menggunakan Microsoft Authenticator
        7. Jika gagal, hubungi helpdesk ext. 1500 untuk manual reset
        Catatan: Password harus minimal 12 karakter, kombinasi huruf besar/kecil, angka, simbol.
        """
    },
    {
        "id": "sop-002",
        "title": "VPN Connection Troubleshooting",
        "category": "network",
        "content": """
        SOP: Troubleshooting koneksi VPN BSI.
        Step 1 - Cek koneksi internet: Pastikan internet aktif sebelum konek VPN.
        Step 2 - Restart VPN client: Tutup Cisco AnyConnect, tunggu 10 detik, buka lagi.
        Step 3 - Clear credentials: Profile → disconnect → hapus saved password → reconnect.
        Step 4 - Cek firewall: Pastikan port 443 dan 4500 tidak diblokir.
        Step 5 - Update VPN client: Versi minimum yang didukung adalah 4.10+
        Step 6 - Jika masih gagal: jalankan 'ipconfig /flushdns' di CMD (Windows) 
                 atau 'sudo dscacheutil -flushcache' (Mac)
        Kontak: helpdesk@bsi.mitsubishi.co.id jika semua langkah gagal.
        """
    },
    {
        "id": "sop-003",
        "title": "ERP SAP Login Error",
        "category": "erp",
        "content": """
        SOP: Menangani error login SAP ERP.
        Error 'Account locked': Akun dikunci setelah 5x salah password.
          → Solusi: Kirim email ke sap-admin@bsi.mitsubishi.co.id dengan subject 'UNLOCK-SAP-[username]'
        Error 'No RFC connection': SAP application server tidak tersedia.
          → Solusi: Cek status di http://sapstatus.bsi.internal, tunggu atau eskalasi ke tim SAP Basis
        Error 'License expired': Lisensi user SAP habis.
          → Solusi: Minta perpanjangan melalui IT Request Form di ServiceNow
        Error 'Transport Request': Ada perubahan konfigurasi pending.
          → Solusi: Tunggu 15 menit setelah deployment selesai, atau restart SAP GUI
        """
    },
    {
        "id": "sop-004",
        "title": "Hardware Request Process",
        "category": "hardware",
        "content": """
        SOP: Prosedur permintaan hardware baru.
        1. Buka ServiceNow Portal: https://bsi.service-now.com
        2. Pilih 'Request Something' → 'Hardware & Peripherals'
        3. Isi form: jenis hardware, justifikasi bisnis, tanggal dibutuhkan
        4. Approval flow: Direct Manager → IT Manager → Procurement (>5 juta IDR)
        5. SLA pengiriman: laptop 5 hari kerja, aksesoris 2 hari kerja
        Catatan: Untuk replacement karena rusak, sertakan foto dan nomor aset.
        Untuk laptop baru hires, submit minimal H-7 sebelum tanggal bergabung.
        """
    },
    {
        "id": "sop-005",
        "title": "Email & Outlook Troubleshooting",
        "category": "access",
        "content": """
        SOP: Masalah umum Outlook & email BSI.
        Outlook tidak bisa kirim/terima email:
          1. Cek status server: File → Account Settings → Exchange account → Test
          2. Hapus cache Outlook: tutup Outlook, hapus folder %localappdata%\Microsoft\Outlook\*.ost
          3. Repair Office: Control Panel → Programs → Microsoft 365 → Repair
        Email quota penuh (mailbox full):
          → Arsipkan email lama ke PST file atau OneDrive
          → Minta penambahan quota via ServiceNow jika perlu
        Tidak bisa akses email dari browser:
          → URL: https://outlook.office365.com, gunakan akun @bsi.mitsubishi.co.id
          → Clear browser cache, coba incognito mode
        """
    },
    {
        "id": "sop-006",
        "title": "Printer & Scanning Issues",
        "category": "hardware",
        "content": """
        SOP: Troubleshooting printer dan scanner.
        Printer tidak terdeteksi:
          1. Restart Print Spooler: services.msc → Print Spooler → Restart
          2. Remove dan re-add printer: Settings → Printers → Remove → Add printer
          3. Reinstall driver dari: http://drivers.bsi.internal/printer
        Print queue stuck:
          1. Stop Print Spooler service
          2. Hapus semua file di C:\Windows\System32\spool\PRINTERS
          3. Start Print Spooler service kembali
        Scanner tidak muncul di PC:
          → Cek kabel USB, pastikan scanner dalam mode 'Scan to PC'
          → Install WIA driver terbaru dari shared drive \\fileserver\IT-Tools\Drivers
        """
    },
]

print(f"SOP docs loaded: {len(SOP_DOCS)} dokumen")

## CHAPTER 2 — Build Retrieval Pipeline (Hour 2)

In [ ]:
# ── STEP 2A: Chunking SOP Documents ────────────────────────────────────
# Chunk ~500 tokens, 50 token overlap

import tiktoken
from typing import List, Dict

def chunk_document(doc: Dict, chunk_size: int = 500, overlap: int = 50) -> List[Dict]:
    """Memotong dokumen menjadi chunk dengan overlap."""
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(doc["content"])
    
    chunks = []
    start = 0
    chunk_idx = 0
    
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = enc.decode(chunk_tokens)
        
        chunks.append({
            "id": f"{doc['id']}-chunk-{chunk_idx}",
            "parent_id": doc["id"],
            "title": doc["title"],
            "category": doc["category"],
            "content": chunk_text.strip(),
            "chunk_index": chunk_idx,
        })
        
        start += chunk_size - overlap
        chunk_idx += 1
    
    return chunks


# Chunk semua dokumen
all_chunks = []
for doc in SOP_DOCS:
    chunks = chunk_document(doc)
    all_chunks.extend(chunks)
    print(f"  {doc['title']}: {len(chunks)} chunk(s)")

print(f"\nTotal chunks: {len(all_chunks)}")

In [ ]:
# ── STEP 2B: Generate Embeddings ────────────────────────────────────────
# Menggunakan text-embedding-3-small via Azure OpenAI

from openai import AzureOpenAI
import time

aoai_client = AzureOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    api_key=AOAI_KEY,
    api_version="2024-02-01"
)

def generate_embeddings(texts: List[str], batch_size: int = 16) -> List[List[float]]:
    """Generate embeddings dalam batch."""
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        response = aoai_client.embeddings.create(
            model=EMBED_DEPLOY,
            input=batch
        )
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"  Batch {i//batch_size + 1}: {len(batch)} embeddings dibuat")
        time.sleep(0.5)  # hindari rate limit
    
    return all_embeddings


# Generate embeddings untuk semua chunks
print("Generating embeddings...")
chunk_texts = [c["content"] for c in all_chunks]
embeddings = generate_embeddings(chunk_texts)

# Tambahkan embedding ke setiap chunk
for chunk, emb in zip(all_chunks, embeddings):
    chunk["content_vector"] = emb

print(f"\nSelesai! {len(embeddings)} embeddings dibuat (dim={len(embeddings[0])})")

In [ ]:
# ── STEP 2C: Create Azure AI Search Index ───────────────────────────────
# Hybrid (keyword + vector) + Semantic Ranker

from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    SemanticConfiguration, SemanticSearch, SemanticPrioritizedFields,
    SemanticField, SimpleField, SearchableField
)
from azure.core.credentials import AzureKeyCredential

index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT,
    credential=AzureKeyCredential(SEARCH_KEY)
)

# Definisi index schema
fields = [
    SimpleField(name="id",         type=SearchFieldDataType.String, key=True),
    SimpleField(name="parent_id",  type=SearchFieldDataType.String, filterable=True),
    SearchableField(name="title",    type=SearchFieldDataType.String),
    SimpleField(name="category",   type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchableField(name="content",  type=SearchFieldDataType.String),
    SimpleField(name="chunk_index",type=SearchFieldDataType.Int32),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,  # text-embedding-3-small dimension
        vector_search_profile_name="myHnswProfile"
    )
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="myHnsw")],
    profiles=[VectorSearchProfile(name="myHnswProfile", algorithm_configuration_name="myHnsw")]
)

semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        content_fields=[SemanticField(field_name="content")]
    )
)
semantic_search = SemanticSearch(configurations=[semantic_config])

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search
)

# Buat atau update index
result = index_client.create_or_update_index(index)
print(f"Index '{result.name}' berhasil dibuat/diupdate ✓")

In [ ]:
# ── STEP 2D: Upload Documents to Search Index ───────────────────────────

from azure.search.documents import SearchClient

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_KEY)
)

# Upload dalam batch
BATCH_SIZE = 100
for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i:i+BATCH_SIZE]
    result = search_client.upload_documents(documents=batch)
    print(f"Batch {i//BATCH_SIZE + 1}: {len([r for r in result if r.succeeded])} dokumen berhasil diupload")

print(f"\nTotal {len(all_chunks)} chunks diupload ke index '{INDEX_NAME}' ✓")

In [ ]:
# ── STEP 2E: RAG Retrieval Function ─────────────────────────────────────
# Hybrid search (keyword + vector) + semantic reranker

from azure.search.documents.models import VectorizableTextQuery, QueryType

def retrieve_knowledge(query: str, top_k: int = 3, category_filter: str = None) -> List[Dict]:
    """
    Hybrid retrieval: keyword + vector + semantic reranker.
    top_k=3 adalah sweet spot untuk cost vs quality (C1: <$0.01/ticket).
    """
    # Generate query embedding
    query_emb = aoai_client.embeddings.create(
        model=EMBED_DEPLOY,
        input=[query]
    ).data[0].embedding
    
    # Build filter
    filter_str = f"category eq '{category_filter}'" if category_filter else None
    
    # Hybrid search dengan semantic reranker
    results = search_client.search(
        search_text=query,
        vector_queries=[
            VectorizableTextQuery(text=query, k_nearest_neighbors=top_k, fields="content_vector")
        ],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="my-semantic-config",
        top=top_k,
        filter=filter_str,
        select=["id", "title", "category", "content", "parent_id"]
    )
    
    docs = []
    for r in results:
        docs.append({
            "id": r["id"],
            "title": r["title"],
            "category": r["category"],
            "content": r["content"],
            "score": r["@search.reranker_score"] or r["@search.score"]
        })
    
    return docs


# Test retrieval
test_docs = retrieve_knowledge("bagaimana cara reset MFA token?")
print(f"Retrieved {len(test_docs)} dokumen:")
for d in test_docs:
    print(f"  [{d['score']:.3f}] {d['title']} ({d['category']})")

In [ ]:
# ── STEP 2F: RAG Responder ───────────────────────────────────────────────
# Prompt templates per kategori + citasi SOP

SYSTEM_PROMPT_TEMPLATE = """Kamu adalah AI assistant helpdesk IT untuk BSI (Mitsubishi Indonesia).
Kategori tiket: {category}

ATURAN PENTING:
1. Jawab HANYA berdasarkan konteks SOP yang diberikan
2. Jika tidak ada info yang relevan, katakan "Maaf, saya tidak menemukan solusi di SOP yang tersedia."
3. Selalu sertakan sumber SOP di akhir jawaban dalam format [Sumber: nama_sop]
4. Gunakan bahasa yang profesional namun ramah
5. Berikan langkah-langkah yang jelas dan mudah diikuti

KONTEKS SOP:
{context}"""

def generate_response(ticket: str, retrieved_docs: List[Dict], category: str = "general") -> Dict:
    """
    Generate respons agent-ready dari tiket + konteks SOP.
    C3: Honest answers — say 'I don't know' when context is empty.
    C4: Cite or it didn't happen — setiap jawaban harus ada sumbernya.
    """
    # C3: Tidak ada konteks relevan
    if not retrieved_docs:
        return {
            "response": "Maaf, saya tidak menemukan solusi di SOP yang tersedia. Silakan eskalasi ke tim IT senior.",
            "sources": [],
            "grounded": False
        }
    
    # Build konteks dari dokumen yang diambil
    context_parts = []
    sources = []
    for i, doc in enumerate(retrieved_docs):
        context_parts.append(f"[SOP-{i+1}: {doc['title']}]\n{doc['content']}")
        if doc['title'] not in sources:
            sources.append(doc['title'])
    
    context = "\n\n".join(context_parts)
    
    # Build prompt
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        category=category,
        context=context
    )
    
    # Generate dengan GPT-4.1-mini
    import time
    start_time = time.time()
    
    completion = aoai_client.chat.completions.create(
        model=CHAT_DEPLOY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Tiket: {ticket}"}
        ],
        max_tokens=600,
        temperature=0.1  # rendah untuk jawaban faktual/konsisten
    )
    
    latency_ms = (time.time() - start_time) * 1000
    
    # Hitung biaya
    usage = completion.usage
    # GPT-4.1-mini pricing (approximate): $0.15/1M input, $0.60/1M output
    cost_usd = (usage.prompt_tokens * 0.00000015) + (usage.completion_tokens * 0.00000060)
    
    return {
        "response": completion.choices[0].message.content,
        "sources": sources,
        "grounded": True,
        "usage": {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens,
            "cost_usd": round(cost_usd, 6),
            "latency_ms": round(latency_ms, 1)
        }
    }


# ── MILESTONE TEST: 'how do I reset my MFA token?' ──
test_ticket = "Halo helpdesk, MFA token saya tidak bisa dipakai login. Bagaimana cara reset?"
retrieved = retrieve_knowledge(test_ticket, top_k=3)
result = generate_response(test_ticket, retrieved, category="access")

print("=" * 60)
print("RESPONS AGENT:")
print(result["response"])
print("\nSumber:", result["sources"])
print("\nMetrik:", result["usage"])
print(f"\nC1 Cost check: ${result['usage']['cost_usd']:.6f} (target: <$0.01) {'✓' if result['usage']['cost_usd'] < 0.01 else '✗'}")
print(f"C2 Latency check: {result['usage']['latency_ms']}ms (target: <4000ms) {'✓' if result['usage']['latency_ms'] < 4000 else '✗'}")

## CHAPTER 3 — Intelligence & Quality (Hour 3)

In [ ]:
# ── STEP 3A: Ticket Classifier ───────────────────────────────────────────
# Training data (labeled tickets) + scikit-learn classifier
# Kemudian di-deploy sebagai Azure ML managed endpoint

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
import joblib

# Mock training data (200+ tickets di production)
TRAINING_DATA = [
    # network
    ("VPN tidak bisa connect padahal internet normal", "network"),
    ("Tidak bisa akses share drive dari rumah", "network"),
    ("Internet kantor sangat lambat hari ini", "network"),
    ("Cannot connect to office WiFi", "network"),
    ("Network drive H: tidak muncul di File Explorer", "network"),
    ("VPN disconnect terus setiap 10 menit", "network"),
    ("Tidak bisa ping ke server internal", "network"),
    # access
    ("Password saya expired, tidak bisa login Windows", "access"),
    ("MFA tidak muncul di handphone, tidak bisa masuk email", "access"),
    ("Akun SAP saya terkunci", "access"),
    ("Tidak bisa login ke portal HR", "access"),
    ("Reset password untuk aplikasi internal", "access"),
    ("2FA / authenticator saya hilang karena ganti HP", "access"),
    ("Username saya tidak dikenali sistem", "access"),
    # hardware
    ("Laptop saya layarnya retak, perlu diganti", "hardware"),
    ("Request mouse dan keyboard baru untuk WFH", "hardware"),
    ("Printer di lantai 3 tidak bisa print", "hardware"),
    ("Butuh laptop baru untuk karyawan baru", "hardware"),
    ("Hard disk laptop saya hampir penuh", "hardware"),
    ("Scanner tidak terdeteksi di komputer", "hardware"),
    ("Headset untuk meeting rusak", "hardware"),
    # erp
    ("SAP error RFC connection failed saat mau buat PO", "erp"),
    ("Tidak bisa approve PR di SAP karena akun terkunci", "erp"),
    ("Error di modul MM SAP saat posting GR", "erp"),
    ("SAP GUI tidak bisa dibuka, license expired", "erp"),
    ("Data di SAP tidak sinkron dengan laporan Excel", "erp"),
    ("Transport request gagal di SAP", "erp"),
    # other
    ("Butuh training penggunaan Teams", "other"),
    ("Pertanyaan tentang kebijakan IT security", "other"),
    ("Cara backup data ke OneDrive", "other"),
    ("Lisensi Office 365 untuk departemen baru", "other"),
    ("Cara share folder di SharePoint", "other"),
]

df = pd.DataFrame(TRAINING_DATA, columns=["text", "label"])
print("Distribusi kelas training:")
print(df["label"].value_counts())

# Train classifier pipeline
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

clf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ("clf",   LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced"))
])

clf_pipeline.fit(X_train, y_train)

# Evaluasi
y_pred = clf_pipeline.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Cross-validation
cv_scores = cross_val_score(clf_pipeline, df["text"], df["label"], cv=5, scoring="f1_macro")
print(f"Cross-val F1 (macro): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Simpan model
joblib.dump(clf_pipeline, "ticket_classifier.pkl")
print("\nModel disimpan: ticket_classifier.pkl ✓")

In [ ]:
# ── STEP 3B: Deploy Classifier ke Azure ML Managed Endpoint ─────────────
# Score script untuk endpoint

SCORE_SCRIPT = '''
import json
import joblib
import os

def init():
    global model
    model_path = os.path.join(os.environ.get("AZUREML_MODEL_DIR", "."), "ticket_classifier.pkl")
    model = joblib.load(model_path)

def run(raw_data):
    data = json.loads(raw_data)
    tickets = data.get("tickets", [])
    
    predictions = []
    for ticket in tickets:
        pred_class = model.predict([ticket])[0]
        pred_proba = model.predict_proba([ticket])[0]
        classes = model.classes_
        
        predictions.append({
            "ticket": ticket,
            "category": pred_class,
            "confidence": round(float(max(pred_proba)), 4),
            "probabilities": {
                cls: round(float(prob), 4) 
                for cls, prob in zip(classes, pred_proba)
            }
        })
    
    return json.dumps({"predictions": predictions})
'''

with open("score.py", "w") as f:
    f.write(SCORE_SCRIPT)
print("score.py dibuat ✓")

# Deploy ke Azure ML (uncomment saat ada Azure ML workspace)
# from azure.ai.ml import MLClient
# from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Model, Environment
# from azure.identity import DefaultAzureCredential
#
# ml_client = MLClient(
#     credential=DefaultAzureCredential(),
#     subscription_id=SUBSCRIPTION_ID,
#     resource_group_name=RESOURCE_GROUP,
#     workspace_name=AML_WORKSPACE
# )
#
# # Register model
# model = Model(name="bsi-ticket-classifier", path="ticket_classifier.pkl")
# registered_model = ml_client.models.create_or_update(model)
#
# # Create endpoint
# endpoint = ManagedOnlineEndpoint(name=AML_ENDPOINT_NAME, auth_mode="key")
# ml_client.online_endpoints.begin_create_or_update(endpoint).result()
#
# # Create deployment
# deployment = ManagedOnlineDeployment(
#     name="default",
#     endpoint_name=AML_ENDPOINT_NAME,
#     model=registered_model,
#     code_path=".",
#     scoring_script="score.py",
#     instance_type="Standard_DS2_v2",
#     instance_count=1
# )
# ml_client.online_deployments.begin_create_or_update(deployment).result()
# print(f"Endpoint '{AML_ENDPOINT_NAME}' deployed ✓")

print("\n(Deploy ke Azure ML: uncomment kode di atas setelah workspace tersedia)")

In [ ]:
# ── STEP 3C: End-to-End Pipeline ─────────────────────────────────────────
# Classifier → Retrieval → Generate

def classify_ticket(ticket_text: str) -> Dict:
    """Classify ticket secara lokal (atau via AML endpoint)."""
    # Local (untuk testing)
    category = clf_pipeline.predict([ticket_text])[0]
    proba = max(clf_pipeline.predict_proba([ticket_text])[0])
    return {"category": category, "confidence": round(float(proba), 4)}
    
    # Via AML endpoint (uncomment di production):
    # import urllib.request, json
    # body = json.dumps({"tickets": [ticket_text]}).encode()
    # req = urllib.request.Request(AML_ENDPOINT_URL, body, {"Authorization": f"Bearer {AML_KEY}"})
    # return json.loads(urllib.request.urlopen(req).read())["predictions"][0]


def process_ticket_end_to_end(ticket_text: str) -> Dict:
    """
    Full pipeline: Ticket → Classify → Retrieve → Generate → Response
    """
    import time
    total_start = time.time()
    
    # Step 1: Classify
    classification = classify_ticket(ticket_text)
    category = classification["category"]
    
    # Step 2: Retrieve (filter by category untuk presisi lebih tinggi)
    docs = retrieve_knowledge(ticket_text, top_k=3, category_filter=category)
    # Fallback: jika tidak ada hasil dengan filter, cari tanpa filter
    if not docs:
        docs = retrieve_knowledge(ticket_text, top_k=3)
    
    # Step 3: Generate
    gen_result = generate_response(ticket_text, docs, category=category)
    
    total_latency = (time.time() - total_start) * 1000
    
    return {
        "ticket": ticket_text,
        "category": category,
        "confidence": classification["confidence"],
        "response": gen_result["response"],
        "sources": gen_result["sources"],
        "grounded": gen_result["grounded"],
        "usage": gen_result.get("usage", {}),
        "total_latency_ms": round(total_latency, 1)
    }


# Test end-to-end
test_cases = [
    "VPN saya tidak bisa connect dari rumah, sudah coba restart tapi tetap gagal",
    "Bagaimana cara reset password Windows saya yang sudah expired?",
    "SAP saya error RFC connection failed setiap kali mau buka",
]

for ticket in test_cases:
    print(f"\nTiket: {ticket[:60]}...")
    result = process_ticket_end_to_end(ticket)
    print(f"Kategori: {result['category']} (conf: {result['confidence']})")
    print(f"Grounded: {result['grounded']} | Sumber: {result['sources']}")
    print(f"Latency: {result['total_latency_ms']}ms | Cost: ${result['usage'].get('cost_usd', 'N/A')}")

In [ ]:
# ── STEP 3D: AI Foundry Evaluation ──────────────────────────────────────
# 20 test queries untuk eval groundedness, relevance, coherence

EVAL_DATASET = [
    {"ticket": "Cara reset MFA token yang hilang karena ganti HP", "expected_category": "access"},
    {"ticket": "VPN Cisco AnyConnect disconnect terus setiap 10 menit", "expected_category": "network"},
    {"ticket": "SAP error 'No RFC connection' saat buka transaksi ME21N", "expected_category": "erp"},
    {"ticket": "Request laptop baru untuk karyawan onboarding minggu depan", "expected_category": "hardware"},
    {"ticket": "Password Windows expired, tidak bisa login komputer kantor", "expected_category": "access"},
    {"ticket": "Printer di lantai 2 stuck, queue tidak bisa dihapus", "expected_category": "hardware"},
    {"ticket": "Tidak bisa akses network drive H: setelah pindah meja", "expected_category": "network"},
    {"ticket": "SAP akun terkunci setelah salah password 5 kali", "expected_category": "erp"},
    {"ticket": "Outlook tidak bisa sync email sejak kemarin", "expected_category": "access"},
    {"ticket": "Butuh scanner untuk departemen Finance", "expected_category": "hardware"},
    {"ticket": "Tidak bisa login portal self-service HR", "expected_category": "access"},
    {"ticket": "VPN timeout setelah idle 30 menit", "expected_category": "network"},
    {"ticket": "SAP license expired untuk user baru", "expected_category": "erp"},
    {"ticket": "Layar laptop retak karena jatuh, perlu klaim asuransi", "expected_category": "hardware"},
    {"ticket": "MFA stuck di loading, tidak keluar kode OTP", "expected_category": "access"},
    {"ticket": "Internet sangat lambat di meeting room lantai 4", "expected_category": "network"},
    {"ticket": "Error transport request di SAP setelah deployment tadi malam", "expected_category": "erp"},
    {"ticket": "Request mouse wireless untuk WFH", "expected_category": "hardware"},
    {"ticket": "Cara backup PST file email ke OneDrive", "expected_category": "other"},
    {"ticket": "Tidak bisa kirim email ke domain eksternal", "expected_category": "access"},
]

print(f"Eval dataset: {len(EVAL_DATASET)} tiket siap")

In [ ]:
# ── STEP 3E: Run Evaluations ─────────────────────────────────────────────
# Groundedness, Relevance, Coherence (LLM-as-judge)

EVAL_SYSTEM_PROMPT = """Kamu adalah evaluator kualitas AI assistant helpdesk.
Beri skor 1-5 untuk setiap dimensi:
- groundedness: Apakah jawaban berdasarkan SOP yang ada? (1=halusinasi, 5=100% dari SOP)
- relevance: Apakah jawaban menjawab pertanyaan tiket? (1=tidak relevan, 5=sangat relevan)
- coherence: Apakah jawaban jelas dan profesional? (1=tidak jelas, 5=sangat jelas)

Balas HANYA dalam JSON: {"groundedness": X, "relevance": X, "coherence": X, "reasoning": "..."}"""

def evaluate_response(ticket: str, response: str, context: str) -> Dict:
    """LLM-as-judge evaluation."""
    eval_prompt = f"""Tiket: {ticket}
Konteks SOP: {context[:500]}...
Jawaban AI: {response}

Evaluasi jawaban di atas."""
    
    completion = aoai_client.chat.completions.create(
        model=CHAT_DEPLOY,
        messages=[
            {"role": "system", "content": EVAL_SYSTEM_PROMPT},
            {"role": "user", "content": eval_prompt}
        ],
        max_tokens=200,
        temperature=0.0,
        response_format={"type": "json_object"}
    )
    
    import json
    try:
        return json.loads(completion.choices[0].message.content)
    except:
        return {"groundedness": 3, "relevance": 3, "coherence": 3, "reasoning": "parse error"}


# Jalankan evaluasi untuk semua test cases
eval_results = []

print("Menjalankan evaluasi 20 tiket...")
for i, test in enumerate(EVAL_DATASET):
    # Generate response
    docs = retrieve_knowledge(test["ticket"], top_k=3)
    context = " | ".join([d["content"][:200] for d in docs])
    gen = generate_response(test["ticket"], docs)
    
    # Evaluate
    scores = evaluate_response(test["ticket"], gen["response"], context)
    
    # Classify
    clf = classify_ticket(test["ticket"])
    
    eval_results.append({
        "ticket": test["ticket"],
        "expected_category": test["expected_category"],
        "predicted_category": clf["category"],
        "category_correct": clf["category"] == test["expected_category"],
        "response": gen["response"][:200] + "...",
        "sources": str(gen["sources"]),
        "grounded": gen["grounded"],
        "groundedness_score": scores.get("groundedness", 3),
        "relevance_score": scores.get("relevance", 3),
        "coherence_score": scores.get("coherence", 3),
        "cost_usd": gen.get("usage", {}).get("cost_usd", 0),
        "latency_ms": gen.get("usage", {}).get("latency_ms", 0),
    })
    print(f"  [{i+1:2d}/20] G:{scores.get('groundedness','-')} R:{scores.get('relevance','-')} C:{scores.get('coherence','-')} | {test['ticket'][:50]}")


# Simpan hasil evaluasi
eval_df = pd.DataFrame(eval_results)
eval_df.to_csv("evaluation_report.csv", index=False)

print("\n=== RINGKASAN EVALUASI ===")
print(f"Groundedness avg : {eval_df['groundedness_score'].mean():.2f}/5 (target ≥4.0 ~ 80%)")
print(f"Relevance avg    : {eval_df['relevance_score'].mean():.2f}/5 (target ≥3.5 ~ 70%)")
print(f"Coherence avg    : {eval_df['coherence_score'].mean():.2f}/5")
print(f"Classifier acc   : {eval_df['category_correct'].mean()*100:.1f}%")
print(f"Avg cost/ticket  : ${eval_df['cost_usd'].mean():.6f} (target <$0.01)")
print(f"Avg latency      : {eval_df['latency_ms'].mean():.0f}ms (target <4000ms)")
print("\nFile disimpan: evaluation_report.csv ✓")

## CHAPTER 4 — Optimize & Cost Report (Hour 4)

In [ ]:
# ── STEP 4A: Cost & Performance Sheet ────────────────────────────────────

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = openpyxl.Workbook()

# ── Sheet 1: Per-Ticket Cost Math
ws1 = wb.active
ws1.title = "Per-Ticket Cost Math"

header_fill = PatternFill("solid", fgColor="1E3A5F")
header_font = Font(color="FFFFFF", bold=True)
thin_border = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"), bottom=Side(style="thin")
)

ws1["A1"] = "BSI IT Helpdesk AI — Per-Ticket Cost Calculation"
ws1["A1"].font = Font(bold=True, size=14)
ws1.merge_cells("A1:E1")

ws1.append([])

# Token cost model
headers = ["Komponen", "Token/Request", "Harga per 1M Token", "Cost per Request ($)", "Keterangan"]
ws1.append(headers)
for col, _ in enumerate(headers, 1):
    cell = ws1.cell(row=3, column=col)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center")

cost_rows = [
    ["Embedding query (text-embedding-3-small)", 50, 0.02, "=B4*C4/1000000", "Query ticket ~50 tokens"],
    ["GPT-4.1-mini input (system + context + ticket)", 800, 0.15, "=B5*C5/1000000", "System 200 + context 500 + ticket 100"],
    ["GPT-4.1-mini output (response)", 300, 0.60, "=B6*C6/1000000", "Respons ~300 tokens"],
    ["Azure AI Search (Basic tier per query)", "-", "-", 0.000025, "$1/40k queries ~ $0.000025/query"],
    ["Azure ML classifier inference", "-", "-", 0.000010, "Sangat murah, instance kecil"],
]

for row in cost_rows:
    ws1.append(row)

ws1.append([])
ws1.append(["TOTAL COST PER TICKET", "", "", "=SUM(D4:D8)", "Target: <$0.01"])
total_row = ws1.max_row
ws1.cell(total_row, 1).font = Font(bold=True)
ws1.cell(total_row, 4).font = Font(bold=True, color="008000")

# Adjust column widths
ws1.column_dimensions["A"].width = 45
ws1.column_dimensions["B"].width = 18
ws1.column_dimensions["C"].width = 20
ws1.column_dimensions["D"].width = 22
ws1.column_dimensions["E"].width = 35


# ── Sheet 2: Eval Results
ws2 = wb.create_sheet("Eval Results")

if len(eval_results) > 0:
    eval_headers = list(eval_results[0].keys())
    ws2.append(eval_headers)
    for col, _ in enumerate(eval_headers, 1):
        cell = ws2.cell(row=1, column=col)
        cell.fill = header_fill
        cell.font = header_font
    
    for r in eval_results:
        ws2.append(list(r.values()))
    
    for col in range(1, len(eval_headers)+1):
        ws2.column_dimensions[get_column_letter(col)].width = 20


# ── Sheet 3: Latency Percentiles
ws3 = wb.create_sheet("Latency & Summary")
ws3["A1"] = "Performance Summary"
ws3["A1"].font = Font(bold=True, size=13)

if len(eval_results) > 0:
    latencies = eval_df["latency_ms"].dropna()
    costs = eval_df["cost_usd"].dropna()
    
    summary = [
        [""],
        ["Metrik", "Nilai", "Target", "Status"],
        ["Groundedness avg", f"{eval_df['groundedness_score'].mean():.2f}/5", "≥4.0", "✓" if eval_df['groundedness_score'].mean()>=4 else "✗"],
        ["Relevance avg", f"{eval_df['relevance_score'].mean():.2f}/5", "≥3.5", "✓" if eval_df['relevance_score'].mean()>=3.5 else "✗"],
        ["Coherence avg", f"{eval_df['coherence_score'].mean():.2f}/5", "≥4.0", "-"],
        ["Classifier accuracy", f"{eval_df['category_correct'].mean()*100:.1f}%", "≥80%", "✓" if eval_df['category_correct'].mean()>=0.8 else "✗"],
        ["p50 latency", f"{latencies.quantile(0.5):.0f} ms", "<4000ms", "✓"],
        ["p95 latency", f"{latencies.quantile(0.95):.0f} ms", "<4000ms", "✓" if latencies.quantile(0.95)<4000 else "✗"],
        ["Avg cost/ticket", f"${costs.mean():.6f}", "<$0.01", "✓" if costs.mean()<0.01 else "✗"],
        ["Max cost/ticket", f"${costs.max():.6f}", "<$0.01", "✓" if costs.max()<0.01 else "✗"],
    ]
    for row in summary:
        ws3.append(row)
    
    ws3.append([])
    ws3.append(["Justifikasi Region & Model"])
    ws3.cell(ws3.max_row, 1).font = Font(bold=True)
    ws3.append(["Region", "Indonesia Central", "Data residency BSI / Mitsubishi Indonesia"])
    ws3.append(["Fallback", "Southeast Asia", "Jika service belum tersedia di Indonesia Central"])
    ws3.append(["Model", "gpt-4.1-mini", "Cost 8x lebih murah dari GPT-4.1, quality 94%"])
    ws3.append(["Embedding", "text-embedding-3-small", "$0.02/1M tokens, cukup untuk SOP internal"])
    ws3.append(["Search tier", "Basic 1 replica", "Cukup untuk workshop, SLA 99.9%"])
    
    ws3.column_dimensions["A"].width = 25
    ws3.column_dimensions["B"].width = 25
    ws3.column_dimensions["C"].width = 45


wb.save("cost_performance_report.xlsx")
print("cost_performance_report.xlsx disimpan ✓")

## STEP TERAKHIR — Verifikasi Semua Constraints

| Constraint | Deskripsi | Target |
|---|---|---|
| C1 Cost ceiling | Per-ticket cost | < $0.01 |
| C2 Latency target | End-to-end | < 4 detik |
| C3 Honest answers | Bilang 'tidak tahu' jika tidak ada SOP | Wajib |
| C4 Cite or it didn't happen | Setiap jawaban ada sumber SOP | Wajib |

In [ ]:
# ── FINAL CHECK ──────────────────────────────────────────────────────────

print("=" * 60)
print("SUBMISSION CHECKLIST")
print("=" * 60)

import os
artifacts = [
    ("mini_project_5_bsi_helpdesk.ipynb", "01 Source code"),
    ("evaluation_report.csv",             "02 Evaluation report"),
    ("cost_performance_report.xlsx",      "03 Cost & performance sheet"),
]

for fname, label in artifacts:
    exists = os.path.exists(fname)
    print(f"  {'✓' if exists else '✗'} {label}: {fname}")

print("  ○ 04 Architecture diagram: buat manual di draw.io / Miro")
print()
print("ZIP command:")
print("  zip mini-project-5-{your-name}.zip \\")
print("    mini_project_5_bsi_helpdesk.ipynb \\")
print("    evaluation_report.csv \\")
print("    cost_performance_report.xlsx \\")
print("    architecture_diagram.png")
print()
print("Upload ke: Google Classroom sebelum 23.59 WIB")